In [5]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

Saving X_test_scaled.csv to X_test_scaled.csv


In [6]:
X_train = pd.read_csv('X_train_scaled.csv')
X_test = pd.read_csv('X_test_scaled.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (455, 30), X_test shape: (114, 30)


In [7]:
!pip install scikit-opt

In [8]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sko.GA import GA

In [9]:
def fitness_function(x):
    selected_features = np.where(x == 1)[0]

    if len(selected_features) == 0:
        return 0.0

    xtrain_sub = X_train.iloc[:, selected_features]
    xtest_sub = X_test.iloc[:, selected_features]

    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(xtrain_sub, y_train)
    preds = clf.predict(xtest_sub)

    acc = accuracy_score(y_test, preds)

    feature_penalty = 0.01 * (len(selected_features) / X_train.shape[1])
    fitness = acc - feature_penalty

    return fitness

In [10]:
def objective_function(x):
    selected_features = np.where(np.array(x) == 1)[0]

    if len(selected_features) == 0:
        return 1.0

    xtrain_sub = X_train.iloc[:, selected_features]
    xtest_sub = X_test.iloc[:, selected_features]

    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(xtrain_sub, y_train)
    preds = clf.predict(xtest_sub)

    error = 1.0 - accuracy_score(y_test, preds)
    feature_ratio = len(selected_features) / X_train.shape[1]

    alpha = 0.99
    beta = 0.01

    cost = (alpha * error) + (beta * feature_ratio)
    return cost

In [11]:
n_features = X_train.shape[1]

ga = GA(func=objective_function,
        n_dim=n_features,
        size_pop=40,
        max_iter=30,
        lb=[0]*n_features,
        ub=[1]*n_features,
        precision=1)

best_x, best_y = ga.run()

print("Optimization complete!")
print(f"Best feature vector (1=keep, 0=discard): {best_x}")

Optimization complete!
Best feature vector (1=keep, 0=discard): [0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1.
 1. 0. 1. 0. 0. 0.]


In [12]:
ga_selected_indices = np.where(np.array(best_x) == 1)[0]
ga_selected_feature_names = X_train.columns[ga_selected_indices]

print(f"Total features selected by GA: {len(ga_selected_indices)} out of {n_features}")
print(f"Selected feature names: list({ga_selected_feature_names})")

X_train_ga = X_train.iloc[:, ga_selected_indices]
X_test_ga = X_test.iloc[:, ga_selected_indices]

final_clf = KNeighborsClassifier(n_neighbors=5)
final_clf.fit(X_train_ga, y_train)
ga_preds = final_clf.predict(X_test_ga)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("\n--- GA MODEL PERFORMANCE ---")
print(f"Accuracy:  {accuracy_score(y_test, ga_preds):.4f}")
print(f"Precision: {precision_score(y_test, ga_preds):.4f}")
print(f"Recall:    {recall_score(y_test, ga_preds):.4f}")
print(f"F1-Score:  {f1_score(y_test, ga_preds):.4f}")

Total features selected by GA: 11 out of 30
Selected feature names: list(Index(['x.smoothness_mean', 'x.compactness_mean', 'x.concavity_mean',
       'x.concave_pts_mean', 'x.symmetry_mean', 'x.fractal_dim_mean',
       'x.texture_se', 'x.concave_pts_se', 'x.area_worst',
       'x.smoothness_worst', 'x.concavity_worst'],
      dtype='object'))

--- GA MODEL PERFORMANCE ---
Accuracy:  0.9825
Precision: 1.0000
Recall:    0.9535
F1-Score:  0.9762


In [13]:
ga_results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Feature Count'],
    'GA_Score': [
        accuracy_score(y_test, ga_preds),
        precision_score(y_test, ga_preds),
        recall_score(y_test, ga_preds),
        f1_score(y_test, ga_preds),
        len(ga_selected_indices)
    ]
})
ga_results.to_csv('ga_performance_results.csv', index=False)
files.download('ga_performance_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>